# Validation and preparation of the MDT data

**Purpose.** This notebook validates the cells and the minimization-of-drive-tests (MDT)
records in `data/raw/`, and writes them to `data/processed/` in the Apache Parquet format.
The notebook does not clean, filter, or change a value. If one rule fails, the notebook stops
and writes no file.

**Inputs.**

| File | Content |
|---|---|
| `data/raw/scenario_config.yaml` | Configuration of the run. The notebook reads the limits of the RSRP report from it. |
| `configs/calibration.yaml` | The notebook reads `seed` and `split.fractions` from it. |
| `data/raw/cells.csv` | One row for each cell |
| `data/raw/mdt.csv` | One row for each MDT record, one `rsrp_<cell>` column for each cell |

**Method.** `src.data.validate_cells` and `src.data.validate_mdt` count the rows that break
each rule. `src.data.mdt_to_long` changes the records to the long format.
`src.data.split_records` divides the records at random into the train, validation, and test
parts. The notebook then writes the Parquet files and reads them again to verify them.

**Outputs.**

| File | Content |
|---|---|
| `data/processed/train.parquet` | The train records: one row for each received (record, cell) pair |
| `data/processed/validation.parquet` | The validation records, in the same format |
| `data/processed/test.parquet` | The test records, in the same format |
| `data/processed/cells.parquet` | The content of `cells.csv` |

**Requirements.** Run `task scenario` first. The notebook does not need a GPU.


## 1. Setup

In [ ]:
import pandas as pd
from omegaconf import OmegaConf

from src.core import MDT_POSITION_COLUMNS, read_cells, rsrp_column
from src.data import SPLIT_NAMES, mdt_to_long, split_records, validate_cells, validate_mdt
from src.utils import DATA_DIR, PROJECT_ROOT, setup_notebook

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
cfg = OmegaConf.load(RAW_DIR / "scenario_config.yaml")
calibration_cfg = OmegaConf.load(PROJECT_ROOT / "configs" / "calibration.yaml")
device = setup_notebook(seed=cfg.seed)

## 2. Input data

`read_cells` reads `cells.csv` and stops if a column is missing. `mdt.csv` is read with the
`timestamp` column as a datetime. Table 1 gives the size of the inputs.

In [ ]:
cells = read_cells(RAW_DIR / "cells.csv")
cells_frame = pd.read_csv(RAW_DIR / "cells.csv")
mdt = pd.read_csv(RAW_DIR / "mdt.csv", parse_dates=["timestamp"])
rsrp_columns = [rsrp_column(cell.name) for cell in cells]

pd.Series(
    {
        "Cells": len(cells),
        "Sites": cells_frame["site"].nunique(),
        "MDT records": len(mdt),
        "Columns of mdt.csv": mdt.shape[1],
        "Position columns": len(MDT_POSITION_COLUMNS),
        "RSRP columns": len(rsrp_columns),
        "Received (record, cell) pairs": int(mdt[rsrp_columns].notna().sum().sum()),
    }
).to_frame("Value").style.set_caption("Table 1. Size of the input data.")

## 3. Validation

Table 2 gives the rules. A rule applies to one row of a file. For each rule, the validation
counts the rows that break it. All counts must be zero.

| Rule | File | Source |
|---|---|---|
| The columns are `timestamp`, `x`, `y`, `z`, and one `rsrp_<cell>` column for each cell | `mdt.csv` | Format of `mdt.csv` (`src.core.mdt`). A failure stops the validation with a `ValueError`. |
| `timestamp` has a datetime type | `mdt.csv` | Format of `mdt.csv`. A failure stops the validation with a `ValueError`. |
| Each cell name is unique | `cells.csv` | The long format identifies a cell by its name. |
| Each cell position is finite | `cells.csv` | The distance from a cell to a UE must exist. |
| Each record has a timestamp | `mdt.csv` | Format of `mdt.csv` |
| Each position is finite | `mdt.csv` | The distance from a cell to a UE must exist. |
| Each record has a minimum of one cell | `mdt.csv` | Rule of the generator: it samples a record again if no cell remains. |
| No RSRP is below `measurement.min_rsrp_dbm` | `mdt.csv` | Rule of the generator: it removes a cell below this value. |
| Each RSRP is a multiple of `measurement.rsrp_resolution_db` | `mdt.csv` | Resolution of the SS-RSRP report (3GPP TS 38.133) |
| No record is a copy of another record | `mdt.csv` | The records are statistically independent. A copy shows an error of the export. |

**Table 2.** Validation rules.

A missing RSRP is not an error. It shows that the UE did not receive the cell, or that the
RSRP was below `measurement.min_rsrp_dbm`.

In [ ]:
results = {
    **validate_cells(cells),
    **validate_mdt(mdt, cells, cfg.measurement.min_rsrp_dbm, cfg.measurement.rsrp_resolution_db),
}
for name, count in results.items():
    print(f"{'PASS' if count == 0 else 'FAIL'}  {name}: {count} rows")
assert all(count == 0 for count in results.values()), "The data is not valid. No file is written."

## 4. Long format

The calibration compares one measured RSRP with one simulated RSRP. The unit of the data is
thus one (record, cell) pair. Let $r$ be the index of a record in `mdt.csv`, and let
$\mathcal{C}_r$ be the set of cells with a reported RSRP in record $r$. The long format has
one row for each pair $(r, c)$ with $c \in \mathcal{C}_r$:

$$\left(r,\ t_r,\ x_r,\ y_r,\ z_r,\ c,\ \widehat{\mathrm{RSRP}}_{r,c}\right).$$

The number of rows is thus $\sum_r |\mathcal{C}_r|$. A pair with a missing RSRP gives no
row. The product of `record` and `cells.parquet` gives all pairs again. Table 3 gives the
columns, and Table 4 gives the first rows.

In [ ]:
long = mdt_to_long(mdt, cells)
assert len(long) == mdt[rsrp_columns].notna().sum().sum()
assert long["record"].nunique() == len(mdt)

descriptions = {
    "record": "Index of the record in mdt.csv, from 0",
    "timestamp": "Start of the time slot of the record",
    "x": "Reported UE position on the x axis of the scene (m)",
    "y": "Reported UE position on the y axis of the scene (m)",
    "z": "UE position on the z axis of the scene (m)",
    "cell": "Name of the cell, as in cells.parquet",
    "rsrp_dbm": "Reported RSRP of the cell (dBm)",
}
display(
    pd.DataFrame(
        {"Type": long.dtypes.astype(str), "Description": pd.Series(descriptions)}
    ).style.set_caption("Table 3. Columns of the long format.")
)
long.head(len(cells)).style.set_caption("Table 4. First rows of the long format.").format(
    precision=1
)

## 5. Split

The calibration learns from one part of the records and measures its error on a different
part. `split_records` puts the records in a random order with the seed `seed`. Let $N$ be
the number of records and let $f_\mathrm{train}$, $f_\mathrm{val}$ be the first two values
of `split.fractions`. The train part gets the first $\lfloor f_\mathrm{train} N \rfloor$
records, the validation part gets the subsequent $\lfloor f_\mathrm{val} N \rfloor$ records,
and the test part gets all other records. All (record, cell) pairs of one record go to the
same part, because they have the same UE position. The `record` values do not change, thus
a row of each part still points to its row of `mdt.csv`. Table 5 gives the size of the parts.

The division is random, as in arXiv:2311.18558, Section IV-C. A test record can thus be
near a train record.

In [ ]:
parts = split_records(long, calibration_cfg.split.fractions, seed=calibration_cfg.seed)
assert list(parts) == list(SPLIT_NAMES)
assert sum(part["record"].nunique() for part in parts.values()) == len(mdt)
assert sum(len(part) for part in parts.values()) == len(long)

pd.DataFrame(
    {
        "Fraction in the configuration": list(calibration_cfg.split.fractions),
        "Records": [part["record"].nunique() for part in parts.values()],
        "(record, cell) pairs": [len(part) for part in parts.values()],
    },
    index=list(parts),
).style.set_caption("Table 5. Size of the parts.")

## 6. Output

The notebook writes one file for each part and one file for the cells. It reads each file
again and compares it with the frame in memory. Table 6 gives the files.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
outputs = {f"{name}.parquet": part for name, part in parts.items()}
outputs["cells.parquet"] = cells_frame
rows = []
for name, frame in outputs.items():
    path = PROCESSED_DIR / name
    frame.to_parquet(path, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(path), frame)
    rows.append(
        {
            "File": path.relative_to(PROJECT_ROOT).as_posix(),
            "Rows": len(frame),
            "Columns": frame.shape[1],
            "Size (kB)": path.stat().st_size / 1e3,
        }
    )
pd.DataFrame(rows).style.set_caption("Table 6. Files in data/processed.").format(
    {"Size (kB)": "{:.1f}"}
).hide(axis="index")

## 7. Summary

All rules of Table 2 have a count of zero, else the notebook stops in Section 3. The
processed files contain the same values as the raw files. Only the layout and the file format
changed: one row for each received (record, cell) pair, in Parquet, with the records divided
into three parts. The calibration notebook reads `data/processed/train.parquet`,
`data/processed/validation.parquet`, `data/processed/test.parquet`, and
`data/processed/cells.parquet`.